# Cross-session sequence dynamics across **all** synapses

This notebook does four things:

1. Imports the patched `analyze_sequence_dynamics` implementation from `analysis_all_synapses.py`.
2. Recomputes sequence tables across sessions **without restricting to image-activated synapses**.
3. Merges the resulting sequence summaries with the cross-session RMS table.
4. Generates depth-, session-order-, and session-type-resolved plots, plus slope-vs-RMS analyses.

Place this notebook in the **same folder** as:
- `analysis_all_synapses.py`
- `activation_summary.csv`
- `rms.csv`

You can later copy `analysis_all_synapses.py` into your repo's `vip_slap2_analysis/glutamate/analysis.py` if you want this behavior to become the default in your package.


In [ ]:
import os
import sys
import traceback
import warnings
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from IPython.display import display
from vip_slap2_analysis.utils.utils import save_figure
from vip_slap2_analysis.io.session_registry import VIPSessionRegistry

sns.set_style('white')
params = {'legend.fontsize': 'x-large',
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))


In [ ]:
%load_ext autoreload
%autoreload 2

%matplotlib notebook

In [ ]:
savepath = r"C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Data_Club\April2026\figures"

In [ ]:
git_dir = Path(r"C:\Users\andrew.shelton\Dropbox\allen institute\Python_Code\ams\ophys\vip-slap2-analysis\src\vip_slap2_analysis")
data_dir = Path(r"C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Data_Club\April2026\data"
)
ANALYSIS_PATH = git_dir / 'glutamate' / 'analysis.py'
ACTIVATION_SUMMARY_PATH = data_dir / 'activation_summary.csv'
RMS_PATH = data_dir / 'rms.csv'
OLD_SEQUENCE_SUMMARY_PATH = data_dir / 'sequence_summary.csv'  # optional

BASE_PATH = r'\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics'

TARGET_MICE = [
    803496,
    804730, 804733, 810196,
    809047, 803121,
    826033, 838410, 834788,
]

EXCLUDE_SESSION_TYPES = ['expression_check', 'volume_imaging']
PARADIGMS = ['change_detection_passive']

SAVE_PER_SESSION_TABLES = False
SAVE_CROSS_SESSION_TABLES = True
CROSS_SESSION_OUTDIR = data_dir / 'sequence_all_synapses_outputs'

# Sequence-analysis settings
SEQUENCE_CONFIG_KWARGS = dict(
    min_images_for_sequence=4,
    min_positions_for_sequence=3,
    sequence_peak_window_samples=10,
    sequence_n_quantile_bins=4,
    sequence_min_count_per_position=2,
    sequence_norm_strategy='r0_abs',
    sequence_rank_by='response_amplitude',
    sequence_response_classes=None,  # IMPORTANT: None => include all synapses
)


In [ ]:
# ------------------------------------------------------------------
# Import patched analysis module from a sibling file
# ------------------------------------------------------------------
# assert ANALYSIS_PATH.exists(), f'Could not find {ANALYSIS_PATH}'
assert ACTIVATION_SUMMARY_PATH.exists(), f'Could not find {ACTIVATION_SUMMARY_PATH}'
assert RMS_PATH.exists(), f'Could not find {RMS_PATH}'

spec = importlib.util.spec_from_file_location('analysis_all_synapses', ANALYSIS_PATH)
analysis_all_synapses = importlib.util.module_from_spec(spec)
sys.modules['analysis_all_synapses'] = analysis_all_synapses
spec.loader.exec_module(analysis_all_synapses)

GlutamateAnalysisConfig = analysis_all_synapses.GlutamateAnalysisConfig
resolve_glutamate_analysis_paths = analysis_all_synapses.resolve_glutamate_analysis_paths
load_session_activation_summary = analysis_all_synapses.load_session_activation_summary
analyze_sequence_dynamics = analysis_all_synapses.analyze_sequence_dynamics

print(f'Imported patched analysis module from: {ANALYSIS_PATH}')


In [ ]:
# ------------------------------------------------------------------
# Colors and small utilities
# ------------------------------------------------------------------
try:
    from PNW_cmap import PNW_cmap
    _, _, cp = PNW_cmap.get_PNW_cmap('Sailboat', n_colors=4)
    DEPTH_PALETTE = {25: cp[::-1][0], 100: cp[::-1][1], 200: cp[::-1][2], 250: cp[::-1][3]}
except Exception:
    pal = sns.color_palette('viridis', 4)
    DEPTH_PALETTE = {25: pal[0], 100: pal[1], 200: pal[2], 250: pal[3]}

DEPTH_ORDER = [25, 100, 200, 250]
SESSION_TYPE_ORDER = ['familiar', 'novel', 'novel+']


def dmd_to_depth(asset, dmd_name):
    dmd_name = str(dmd_name).upper()
    if dmd_name == 'DMD1':
        return asset.metadata.get('dmd1_depth', np.nan)
    if dmd_name == 'DMD2':
        return asset.metadata.get('dmd2_depth', np.nan)
    return np.nan


def add_asset_metadata(df, asset):
    if df is None or len(df) == 0:
        return df
    out = df.copy()
    out['session_order'] = asset.metadata.get('session_#', np.nan)
    out['session_type'] = asset.metadata.get('session_type', np.nan)
    out['dmd1_depth'] = asset.metadata.get('dmd1_depth', np.nan)
    out['dmd2_depth'] = asset.metadata.get('dmd2_depth', np.nan)
    out['depth_um'] = out['dmd'].map(lambda x: dmd_to_depth(asset, x))
    return out


def style_axis(ax):
    sns.despine(ax=ax)
    ax.tick_params(axis='x', which='major', top=False, labelsize=11)
    ax.tick_params(axis='y', which='major', right=False, labelsize=11)
    for spine in ['left', 'bottom']:
        if spine in ax.spines:
            ax.spines[spine].set_linewidth(1.8)


def sem(x):
    x = pd.Series(x).dropna().to_numpy()
    if len(x) == 0:
        return np.nan
    return np.std(x, ddof=1) / np.sqrt(len(x)) if len(x) > 1 else 0.0


In [ ]:
target_mice = [
    803496,
    804730,804733,810196,
    809047,803121,
    826033,838410,834788
]

registry = VIPSessionRegistry.from_basepath(
    r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics'
)

process_df = registry.sessions(
    subject_ids=target_mice,
    exclude_session_types=["expression_check", "volume_imaging"],
    paradigms=["change_detection_passive"],
)

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

print(f"Loaded {len(assets)} session assets")

In [ ]:
process_df.head()

In [ ]:
# ------------------------------------------------------------------
# Load cross-session activation and RMS tables
# ------------------------------------------------------------------
activation_master = pd.read_csv(ACTIVATION_SUMMARY_PATH)
rms_master = pd.read_csv(RMS_PATH)

for df in [activation_master, rms_master]:
    if 'Unnamed: 0' in df.columns:
        df.drop(columns=['Unnamed: 0'], inplace=True)
    for col in ['session_id', 'dmd', 'synapse_id']:
        if col in df.columns:
            df[col] = df[col].astype(str)

print('activation rows:', len(activation_master))
print('rms rows:', len(rms_master))
print('image-family activation rows:', (activation_master['stimulus_family'] == 'image').sum())

display(activation_master.head())
display(rms_master.head())


In [ ]:
# ------------------------------------------------------------------
# Recompute sequence tables across sessions using ALL synapses
# ------------------------------------------------------------------
config = GlutamateAnalysisConfig(**SEQUENCE_CONFIG_KWARGS)

sequence_position_all = []
sequence_per_image_all = []
sequence_summary_all = []
failures = []

for asset in assets:
    print(asset.session_id)
    try:
        paths = resolve_glutamate_analysis_paths(asset.session_dir)
        if not paths.sequence_npz.exists():
            failures.append((asset.session_id, 'missing sequence_npz'))
            continue

        session_activation = load_session_activation_summary(
            paths.single_trial_npz,
            activation_summary=activation_master,
        )

        seq_pos, seq_per_im, seq_sum = analyze_sequence_dynamics(
            paths.sequence_npz,
            activation_summary_df=session_activation,
            config=config,
            tuning_per_image_df=None,
            tuning_summary_df=None,
        )

        seq_pos = add_asset_metadata(seq_pos, asset)
        seq_per_im = add_asset_metadata(seq_per_im, asset)
        seq_sum = add_asset_metadata(seq_sum, asset)

        sequence_position_all.append(seq_pos)
        sequence_per_image_all.append(seq_per_im)
        sequence_summary_all.append(seq_sum)

        if SAVE_PER_SESSION_TABLES:
            outdir = asset.derived_dir / 'glutamate' / 'glutamate_analysis'
            outdir.mkdir(parents=True, exist_ok=True)
            seq_pos.to_csv(outdir / 'sequence_position_table_all_synapses.csv', index=False)
            seq_per_im.to_csv(outdir / 'sequence_per_image_table_all_synapses.csv', index=False)
            seq_sum.to_csv(outdir / 'sequence_summary_table_all_synapses.csv', index=False)

    except Exception as exc:
        failures.append((asset.session_id, repr(exc)))
        traceback.print_exc()

sequence_position_all = pd.concat(sequence_position_all, ignore_index=True) if sequence_position_all else pd.DataFrame()
sequence_per_image_all = pd.concat(sequence_per_image_all, ignore_index=True) if sequence_per_image_all else pd.DataFrame()
sequence_summary_all = pd.concat(sequence_summary_all, ignore_index=True) if sequence_summary_all else pd.DataFrame()

print('sequence_position_all:', sequence_position_all.shape)
print('sequence_per_image_all:', sequence_per_image_all.shape)
print('sequence_summary_all:', sequence_summary_all.shape)
print('failures:', len(failures))
if failures:
    display(pd.DataFrame(failures, columns=['session_id', 'error']).head(20))


In [ ]:
# ------------------------------------------------------------------
# Merge RMS metadata and derive slope metrics
# ------------------------------------------------------------------
merge_keys = ['session_id', 'dmd', 'synapse_id']

summary_all = sequence_summary_all.merge(
    rms_master,
    on=merge_keys,
    how='left',
    suffixes=('', '_rms'),
)

per_image_all = sequence_per_image_all.merge(
    rms_master,
    on=merge_keys,
    how='left',
    suffixes=('', '_rms'),
)

for df in [summary_all, per_image_all]:
    if 'depth_um_rms' in df.columns and 'depth_um' in df.columns:
        df['depth_um'] = df['depth_um'].fillna(df['depth_um_rms'])
    elif 'depth_um_rms' in df.columns:
        df.rename(columns={'depth_um_rms': 'depth_um'}, inplace=True)

    if 'session_type_rms' in df.columns and 'session_type' in df.columns:
        df['session_type'] = df['session_type'].fillna(df['session_type_rms'])
    elif 'session_type_rms' in df.columns:
        df.rename(columns={'session_type_rms': 'session_type'}, inplace=True)

    if 'session_order_rms' in df.columns and 'session_order' in df.columns:
        df['session_order'] = df['session_order'].fillna(df['session_order_rms'])
    elif 'session_order_rms' in df.columns:
        df.rename(columns={'session_order_rms': 'session_order'}, inplace=True)

summary_all['slope_direction'] = np.where(
    summary_all['median_overall_slope'] > 0,
    'facilitating',
    np.where(summary_all['median_overall_slope'] < 0, 'adapting', 'flat')
)
summary_all['slope_magnitude'] = summary_all['median_overall_slope']
summary_all['log10_image_rms_dff'] = np.log10(summary_all['image_rms_dff'].where(summary_all['image_rms_dff'] > 0))
summary_all['log10_image_fve_curve'] = np.log10(summary_all['image_fve_curve'].where(summary_all['image_fve_curve'] > 0))

print(summary_all.shape)
display(summary_all.head())


In [ ]:
# ------------------------------------------------------------------
# Save concatenated cross-session tables
# ------------------------------------------------------------------
if SAVE_CROSS_SESSION_TABLES:
    CROSS_SESSION_OUTDIR.mkdir(parents=True, exist_ok=True)
    sequence_position_all.to_csv(CROSS_SESSION_OUTDIR / 'sequence_position_all_synapses.csv', index=False)
    sequence_per_image_all.to_csv(CROSS_SESSION_OUTDIR / 'sequence_per_image_all_synapses.csv', index=False)
    sequence_summary_all.to_csv(CROSS_SESSION_OUTDIR / 'sequence_summary_all_synapses.csv', index=False)
    summary_all.to_csv(CROSS_SESSION_OUTDIR / 'sequence_summary_all_synapses_merged_with_rms.csv', index=False)
    print(f'Saved cross-session tables to: {CROSS_SESSION_OUTDIR}')


In [ ]:
# ------------------------------------------------------------------
# QC: how many synapses were recovered relative to the old activated-only tables?
# ------------------------------------------------------------------
image_activation = (
    activation_master
    .query("stimulus_family == 'image'")
    .groupby(['session_id', 'dmd'])['synapse_id']
    .nunique()
    .rename('n_activation_image_synapses')
    .reset_index()
)

new_seq_counts = (
    sequence_summary_all
    .groupby(['session_id', 'dmd'])['synapse_id']
    .nunique()
    .rename('n_new_sequence_synapses')
    .reset_index()
)

count_compare = image_activation.merge(new_seq_counts, on=['session_id', 'dmd'], how='left')
count_compare['n_new_sequence_synapses'] = count_compare['n_new_sequence_synapses'].fillna(0)
count_compare['fraction_recovered_vs_activation'] = (
    count_compare['n_new_sequence_synapses'] / count_compare['n_activation_image_synapses']
)

if OLD_SEQUENCE_SUMMARY_PATH.exists():
    old_seq_summary = pd.read_csv(OLD_SEQUENCE_SUMMARY_PATH)
    if 'Unnamed: 0' in old_seq_summary.columns:
        old_seq_summary = old_seq_summary.drop(columns=['Unnamed: 0'])
    for col in ['session_id', 'dmd', 'synapse_id']:
        old_seq_summary[col] = old_seq_summary[col].astype(str)

    old_counts = (
        old_seq_summary
        .groupby(['session_id', 'dmd'])['synapse_id']
        .nunique()
        .rename('n_old_sequence_synapses')
        .reset_index()
    )
    count_compare = count_compare.merge(old_counts, on=['session_id', 'dmd'], how='left')
    count_compare['fraction_old_vs_activation'] = (
        count_compare['n_old_sequence_synapses'] / count_compare['n_activation_image_synapses']
    )

print('Median fraction recovered:', np.nanmedian(count_compare['fraction_recovered_vs_activation']))
display(count_compare.sort_values('fraction_recovered_vs_activation').head(20))


## Plot 1 — Slope distributions by depth

This is the cleanest population-level view of whether sequence slopes differ with cortical depth when **all** synapses are included.


In [ ]:
plot_df = summary_all.dropna(subset=['depth_um', 'median_overall_slope']).copy()
# plot_df = plot_df[plot_df['response_class']=='activated']
plot_df['depth_um'] = plot_df['depth_um'].astype(int)
plot_df = plot_df[plot_df['depth_um'].isin(DEPTH_ORDER)]

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
sns.stripplot(
    data=plot_df,
    x='depth_um',
    y='median_overall_slope',
    hue='depth_um',
    order=DEPTH_ORDER,
    hue_order=DEPTH_ORDER,
    palette=DEPTH_PALETTE,
    dodge=False,
    size=3,
    alpha=0.55,
    ax=ax,
)
medians = plot_df.groupby('depth_um')['median_overall_slope'].median().reindex(DEPTH_ORDER)
ax.plot(range(len(DEPTH_ORDER)), medians.values, color='k', lw=2.5, marker='o', zorder=10)
ax.axhline(0, color='k', ls='--', lw=1)
ax.set_xlabel('Depth (µm from pia)',fontsize=15)
ax.set_ylabel('Median sequence slope',fontsize=15)
# ax.set_ylim(-10,10)
ax.legend_.remove()
style_axis(ax)
fig.tight_layout()


## Plot 2 — Slope by session order and depth

This addresses whether facilitation/adaptation changes over the progression of familiar → novel → novel+ experience.


In [ ]:
session_plot = (
    plot_df
    .dropna(subset=['session_order'])
    .groupby(['depth_um', 'session_order'], as_index=False)
    .agg(
        mean_slope=('median_overall_slope', 'mean'),
        sem_slope=('median_overall_slope', sem),
        n=('median_overall_slope', 'size'),
    )
)
session_plot = session_plot[(session_plot['session_order']>1)]
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
for depth in DEPTH_ORDER:
    sub = session_plot[session_plot['depth_um'] == depth].sort_values('session_order')
    if len(sub) == 0:
        continue
    ax.plot(sub['session_order'], sub['mean_slope'], marker='o', lw=2, color=DEPTH_PALETTE[depth], label=f'{depth} µm')
    ax.fill_between(
        sub['session_order'],
        sub['mean_slope'] - sub['sem_slope'],
        sub['mean_slope'] + sub['sem_slope'],
        color=DEPTH_PALETTE[depth],
        alpha=0.18,
    )
ax.set_xticks(np.arange(2,8))
ax.set_xticklabels(['F','F','F','N','N+','N+'],fontsize=15)    

# ax.axhline(0, color='k', ls='--', lw=1)
ax.set_xlabel('Session order',fontsize=15)
ax.set_ylabel('Median sequence slope',fontsize=15)
ax.set_title('Response slope across sessions',fontsize=18)
ax.legend(frameon=False,fontsize=12)
style_axis(ax)
fig.tight_layout()


## Plot 3 — Slope by session type and depth

This gives the categorical version of the same question.


In [ ]:
type_df = plot_df.copy()
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.pointplot(
    data=type_df,
    x='session_type',
    y='median_overall_slope',
    hue='depth_um',
    order=SESSION_TYPE_ORDER,
    hue_order=DEPTH_ORDER,
    palette=DEPTH_PALETTE,
    errorbar='se',
    dodge=0.0,
    markers='o',
    linestyles='-',
    ax=ax,
)
ax.axhline(0, color='k', ls='--', lw=1)
ax.set_xlabel('Session type')
ax.set_ylabel('Median sequence slope')
ax.legend(title='Depth', frameon=False)
style_axis(ax)
fig.tight_layout()


## Plot 4 — Relationship between sequence slope and image modulation strength

Two complementary views are worth checking:

- **Signed slope** vs RMS: are more image-modulated synapses preferentially facilitating or adapting?
- **Absolute slope magnitude** vs RMS: are more image-modulated synapses simply *more dynamic*, regardless of direction?


In [ ]:
summary_all.keys()

In [ ]:
corr_df = summary_all.dropna(subset=['image_rms_dff', 'median_overall_slope', 'depth_um','response_class']).copy()
# corr_df = corr_df[np.abs(corr_df['median_overall_slope'])>1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.scatterplot(
    data=corr_df,
    x='image_rms_dff',
    y='median_overall_slope',
    hue='depth_um',
    hue_order=DEPTH_ORDER,
    palette=DEPTH_PALETTE,
    s=22,
    alpha=0.75,
    edgecolor='none',
    ax=axes[0],
)
# axes[0].axhline(0, color='k', ls='--', lw=1)
axes[0].set_xscale('log')
axes[0].set_xlabel('Image RMS dF/F (log scale)')
axes[0].set_ylabel('Median sequence slope')
axes[0].legend(title='Depth', frameon=False)
style_axis(axes[0])

sns.scatterplot(
    data=corr_df.assign(slope_magnitude=abs(corr_df['median_overall_slope'])),
    x='image_rms_dff',
    y='slope_magnitude',
    hue='depth_um',
    hue_order=DEPTH_ORDER,
    palette=DEPTH_PALETTE,
    s=22,
    alpha=0.75,
    edgecolor='none',
    ax=axes[1],
)
axes[1].set_xscale('log')
axes[1].set_xlabel('Image RMS dF/F (log scale)')
axes[1].set_ylabel('|Median sequence slope|')
axes[1].legend_.remove()

# for ax in axes.flatten():
#     ax.set_ylim(-10,10)
#     ax.set_xlim(1e-2,1e0)

style_axis(axes[1])

fig.tight_layout()


In [ ]:
# Same question using image FVE instead of image RMS dF/F
corr_fve_df = summary_all.dropna(subset=['image_fve_curve', 'median_overall_slope', 'depth_um']).copy()
corr_fve_df = corr_fve_df[corr_fve_df['image_fve_curve'] > 0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.scatterplot(
    data=corr_fve_df,
    x='image_fve_curve',
    y='median_overall_slope',
    hue='depth_um',
    hue_order=DEPTH_ORDER,
    palette=DEPTH_PALETTE,
    s=22,
    alpha=0.75,
    edgecolor='none',
    ax=axes[0],
)
axes[0].axhline(0, color='k', ls='--', lw=1)
axes[0].set_xscale('log')
axes[0].set_xlabel('Image FVE (log scale)')
axes[0].set_ylabel('Median sequence slope')
axes[0].legend(title='Depth', frameon=False)
style_axis(axes[0])

sns.scatterplot(
    data=corr_fve_df.assign(slope_magnitude=np.abs(corr_fve_df['median_overall_slope'])),
    x='image_fve_curve',
    y='slope_magnitude',
    hue='depth_um',
    hue_order=DEPTH_ORDER,
    palette=DEPTH_PALETTE,
    s=22,
    alpha=0.75,
    edgecolor='none',
    ax=axes[1],
)
axes[1].set_xscale('log')
axes[1].set_xlabel('Image FVE (log scale)')
axes[1].set_ylabel('|Median sequence slope|')
axes[1].legend_.remove()
style_axis(axes[1])

fig.tight_layout()


## Correlation tables

These are handy for reporting whether image modulation tracks **slope direction** or **slope magnitude**, overall and within each depth.


In [ ]:
def spearman_table(df, x_col, y_col, group_col=None):
    rows = []
    if group_col is None:
        work = [('all', df)]
    else:
        work = list(df.groupby(group_col))
    for group_name, sub in work:
        sub = sub.dropna(subset=[x_col, y_col])
        if len(sub) < 3:
            continue
        rho, p = spearmanr(sub[x_col], sub[y_col], nan_policy='omit')
        rows.append({
            'group': group_name,
            'n': len(sub),
            'spearman_rho': rho,
            'p_value': p,
        })
    return pd.DataFrame(rows)

rho_signed_rms = spearman_table(corr_df, 'image_rms_dff', 'median_overall_slope')
rho_abs_rms = spearman_table(corr_df.assign(slope_magnitude=np.abs(corr_df['median_overall_slope'])), 'image_rms_dff', 'slope_magnitude')
rho_signed_rms_by_depth = spearman_table(corr_df, 'image_rms_dff', 'median_overall_slope', 'depth_um')
rho_abs_rms_by_depth = spearman_table(corr_df.assign(slope_magnitude=np.abs(corr_df['median_overall_slope'])), 'image_rms_dff', 'slope_magnitude', 'depth_um')

print('Signed slope vs RMS')
display(rho_signed_rms)
display(rho_signed_rms_by_depth)

print('Slope magnitude vs RMS')
display(rho_abs_rms)
display(rho_abs_rms_by_depth)


In [ ]:
# Optional: preferred summary split by response class from activation analysis
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.boxplot(
    data=plot_df,
    x='response_class',
    y='median_overall_slope',
    order=['activated', 'no_change', 'deactivated'],
    showcaps=False,
    showfliers=False,
    boxprops={'facecolor': 'white', 'edgecolor': 'black'},
    whiskerprops={'color': 'black'},
    medianprops={'color': 'black'},
    ax=ax,
)
sns.stripplot(
    data=plot_df,
    x='response_class',
    y='median_overall_slope',
    order=['activated', 'no_change', 'deactivated'],
    hue='depth_um',
    hue_order=DEPTH_ORDER,
    palette=DEPTH_PALETTE,
    dodge=False,
    size=3,
    alpha=0.5,
    ax=ax,
)
ax.axhline(0, color='k', ls='--', lw=1)
ax.set_xlabel('Image response class')
ax.set_ylabel('Median sequence slope')
ax.legend(title='Depth', frameon=False, bbox_to_anchor=(1.02, 1), loc='upper left')
style_axis(ax)
fig.tight_layout()


## Notes

A few practical interpretations to keep in mind:

- `median_overall_slope > 0` means the synapse tends to **facilitate** across repeated image presentations.
- `median_overall_slope < 0` means the synapse tends to **adapt**.
- `image_rms_dff` and `image_fve_curve` are both synapse-level measures of how strongly image identity modulates the response curve.
- If the **signed** slope correlation is weak but the **absolute** slope correlation is strong, that suggests stronger image modulation is associated with **more sequence dynamics**, but not preferentially with facilitation or adaptation.


In [ ]:
import numpy as np
import pandas as pd

# Inputs assumed to already be loaded:
# sequence_per_image_all_synapses
# sequence_summary_all_synapses   # optional but recommended

per_image = per_image_all.copy()
# summary_all = summary_all.copy() if "sequence_summary_all_synapses" in globals() else None

# Core synapse identifier
synapse_keys = [c for c in ["session_id", "subject_id", "dmd", "synapse_id"] if c in per_image.columns]

# Compute magnitude of image-specific slope
per_image["abs_overall_slope"] = per_image["early_slope"].abs()

# For each synapse, pick the image with the largest absolute slope
idx = (
    per_image
    .groupby(synapse_keys, dropna=False)["abs_overall_slope"]
    .idxmax()
)

max_mag_per_synapse = per_image.loc[idx].copy()

# Rename image-specific columns so they are clearly tied to the max-magnitude image
rename_map = {
    "stimulus_name": "max_mag_stimulus_name",
    "stimulus_label": "max_mag_stimulus_label",
    "overall_slope": "max_mag_overall_slope",
    "abs_overall_slope": "max_mag_abs_overall_slope",
    "early_slope": "max_mag_early_slope",
    "late_slope": "max_mag_late_slope",
    "sequence_label": "max_mag_sequence_label",
    "r0": "max_mag_r0",
    "rlast": "max_mag_rlast",
    "rterminal": "max_mag_rterminal",
    "terminal_minus_last": "max_mag_terminal_minus_last",
    "adaptation_index": "max_mag_adaptation_index",
    "early_mean": "max_mag_early_mean",
    "late_mean": "max_mag_late_mean",
    "early_minus_late": "max_mag_early_minus_late",
    "n_positions": "max_mag_n_positions",
    "n_sequences": "max_mag_n_sequences",
}
max_mag_per_synapse = max_mag_per_synapse.rename(columns=rename_map)

# If depth is not already present in the per-image-selected table, pull it from summary_all
if "depth" not in max_mag_per_synapse.columns and summary_all is not None and "depth_um" in summary_all.columns:
    depth_df = summary_all[synapse_keys + ["depth_um"]].drop_duplicates(subset=synapse_keys)
    max_mag_per_synapse = max_mag_per_synapse.merge(
        depth_df,
        on=synapse_keys,
        how="left",
    )

# Keep only useful columns
preferred_cols = [
    *synapse_keys,
    "depth",
    "session_type",
    "session_order",
    "response_class",
    "max_mag_stimulus_name",
    "max_mag_stimulus_label",
    "max_mag_overall_slope",
    "max_mag_abs_overall_slope",
    "max_mag_early_slope",
    "max_mag_late_slope",
    "max_mag_sequence_label",
    "max_mag_r0",
    "max_mag_rlast",
    "max_mag_rterminal",
    "max_mag_terminal_minus_last",
    "max_mag_adaptation_index",
    "max_mag_early_mean",
    "max_mag_late_mean",
    "max_mag_early_minus_late",
    "max_mag_n_positions",
    "max_mag_n_sequences",
]
preferred_cols = [c for c in preferred_cols if c in max_mag_per_synapse.columns]
max_mag_per_synapse = max_mag_per_synapse[preferred_cols].copy()

# Optionally merge in synapse-level summary metrics
if summary_all is not None:
    summary_cols_wanted = [
        *synapse_keys,
        "depth_um",
        "n_images_with_sequences",
        "median_overall_slope",
        "median_early_slope",
        "median_late_slope",
        "median_adaptation_index",
        "median_r0",
        "median_rlast",
        "median_rterminal",
        "median_terminal_minus_last",
        "median_early_minus_late",
        "seq_p",
        "seq_q",
        "sequence_class",
        "log10_image_rms_dff"
    ]
    summary_cols_wanted = [c for c in summary_cols_wanted if c in summary_all.columns]

    max_mag_per_synapse = max_mag_per_synapse.merge(
        summary_all[summary_cols_wanted].drop_duplicates(subset=synapse_keys),
        on=synapse_keys,
        how="left",
        suffixes=("", "_summary"),
    )

    # Resolve duplicate depth columns if both were present
    if "depth_summary" in max_mag_per_synapse.columns:
        if "depth" in max_mag_per_synapse.columns:
            max_mag_per_synapse["depth"] = max_mag_per_synapse["depth"].fillna(max_mag_per_synapse["depth_summary"])
            max_mag_per_synapse = max_mag_per_synapse.drop(columns=["depth_summary"])
        else:
            max_mag_per_synapse = max_mag_per_synapse.rename(columns={"depth_summary": "depth"})

# Sort by magnitude for easy inspection
max_mag_per_synapse = max_mag_per_synapse.sort_values(
    "max_mag_abs_overall_slope",
    ascending=False,
).reset_index(drop=True)

display(max_mag_per_synapse.head(20))

# Optional save
# max_mag_per_synapse.to_csv("sequence_max_magnitude_image_per_synapse.csv", index=False)

In [ ]:
# dft = max_mag_per_synapse[max_mag_per_synapse['response_class']=='activated']
# dft=max_mag_per_synapse[max_mag_per_synapse['max_mag_abs_overall_slope']>1]
dft = max_mag_per_synapse.copy()

In [ ]:
session_plot = (
    dft
    .dropna(subset=['session_order'])
    .groupby(['depth_um', 'session_order'], as_index=False)
    .agg(
        mean_slope=('max_mag_early_slope', 'mean'),
        sem_slope=('max_mag_early_slope', sem),
        n=('max_mag_early_slope', 'size'),
    )
)
session_plot = session_plot[(session_plot['session_order']>1)
#                             &(session_plot['session_order']<6)
#                             &(session_plot['response_class']=='activated')
#                            &(session_plot['mean_slope']>0)
                           ]
fig, ax = plt.subplots(figsize=(6, 4))

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
for depth in DEPTH_ORDER:
    sub = session_plot[session_plot['depth_um'] == depth].sort_values('session_order')
    if len(sub) == 0:
        continue
    ax.plot(sub['session_order'], sub['mean_slope']*1, marker='o', lw=2, color=DEPTH_PALETTE[depth], label=f'{depth} µm')
    ax.fill_between(
        sub['session_order'],
        sub['mean_slope']*1 - sub['sem_slope']/1,
        sub['mean_slope']*1 + sub['sem_slope']/1,
        color=DEPTH_PALETTE[depth],
        alpha=0.18,
    )
ax.set_xticks(np.arange(2,8))
ax.set_xticklabels(['F','F','F','N','N+','N+'],fontsize=15)    

# ax.axhline(0, color='k', ls='--', lw=1)
ax.set_xlabel('Session type')
ax.set_ylabel('Response slope')
ax.legend(frameon=False, fontsize=12)
style_axis(ax)
fig.tight_layout()
filen = 'Session_slope'
save_figure(fig,os.path.join(savepath,filen),formats = ['.pdf','.png'],dpi= 300)

In [ ]:
session_plot = (
    dft
    .dropna(subset=['session_order'])
    .groupby(['depth_um', 'session_order'], as_index=False)
    .agg(
        mean_slope=('max_mag_early_slope', 'mean'),
        sem_slope=('max_mag_early_slope', sem),
        n=('max_mag_early_slope', 'size'),
    )
)
session_plot = session_plot[(session_plot['session_order']>3)
                            &(session_plot['session_order']<7)
#                             &(session_plot['response_class']=='activated')
#                            &(session_plot['mean_slope']>0)
                           ]
fig, ax = plt.subplots(figsize=(4, 4))

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
for depth in DEPTH_ORDER:
    sub = session_plot[session_plot['depth_um'] == depth].sort_values('session_order')
    if len(sub) == 0:
        continue
    ax.plot(sub['session_order'], sub['mean_slope']*1, marker='o', lw=2, color=DEPTH_PALETTE[depth], label=f'{depth} µm')
    ax.fill_between(
        sub['session_order'],
        sub['mean_slope']*1 - sub['sem_slope']/2,
        sub['mean_slope']*1 + sub['sem_slope']/2,
        color=DEPTH_PALETTE[depth],
        alpha=0.18,
    )
ax.set_xticks(np.arange(4,7))
ax.set_xticklabels(['F','N','N+'],fontsize=15)    

# ax.axhline(0, color='k', ls='--', lw=1)
ax.set_xlabel('Session type')
ax.set_ylabel('Response slope')
ax.legend(frameon=False,fontsize=12)
style_axis(ax)
fig.tight_layout()
filen = 'Session_slope_inset'
save_figure(fig,os.path.join(savepath,filen),formats = ['.pdf','.png'],dpi= 300)

In [ ]:
plot_df.keys()

In [ ]:
plot_df = summary_all.dropna(subset=['depth_um', 'median_early_slope']).copy()
plot_df = plot_df[plot_df['response_class']=='activated']
plot_df['depth_um'] = plot_df['depth_um'].astype(int)
plot_df = plot_df[plot_df['depth_um'].isin(DEPTH_ORDER)]

fig, ax = plt.subplots(figsize=(4, 4))
sns.despine()

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

# plot_dft = dft

sns.stripplot(
    data=plot_df,
    x='depth_um',
    y='median_early_slope',
    hue='depth_um',
    order=DEPTH_ORDER,
    hue_order=DEPTH_ORDER,
    palette=DEPTH_PALETTE,
    dodge=False,
    size=5,
    alpha=0.75,
    ax=ax,
    legend=False
    
)
medians = dft.groupby('depth_um')['max_mag_early_slope'].mean().reindex(DEPTH_ORDER)
# ax.plot(range(len(DEPTH_ORDER)), medians.values, color='k', lw=2.5, marker='o', zorder=10)

# ax.set_ylim(-10,10)
ax.set_xlabel('Depth (\u03BCm) from pia')
ax.set_ylabel('Response Slope')
# ax.set_ylim(-80,80)
for spine in ['left','bottom']:
    ax.spines[spine].set_linewidth(2)
ax.set_title('Average Response Slope')
fig.tight_layout()

filen = 'Avg_resp_slope_depth'
save_figure(fig,os.path.join(savepath,filen),formats=['.pdf','.png'],dpi=300)

In [ ]:
# Symmetric-quantile version:
# negative bins = adapting, positive bins = facilitating
# set n_side_bins=5 for 5 adapting bins + 5 facilitating bins

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------
# user-adjustable params
# -----------------------
slope_col = "median_early_slope"   # e.g. "median_overall_slope", "median_late_slope", "median_early_slope"
selectivity_col = "image_fve_curve"
depth_col = "depth_um"

n_side_bins = 5                      # number of bins on each side of zero
min_bin_n = 10
use_log_selectivity = False
include_zero_bin = True             # set True if you want exact-zero slopes shown separately
zero_eps = 1e-12                     # tolerance for treating values as zero
band_quantiles = (0.40, 0.60)        # inner band; try (0.25, 0.75) for IQR
figsize = (6, 4)

# -----------------------
# load from your existing table
# -----------------------
plot_df = summary_all.dropna(subset=[depth_col, slope_col, selectivity_col]).copy()
plot_df = plot_df[plot_df["response_class"] == "activated"].copy()
plot_df[depth_col] = plot_df[depth_col].astype(int)
plot_df = plot_df[plot_df[depth_col].isin(DEPTH_ORDER)].copy()
plot_df = plot_df[np.isfinite(plot_df[slope_col]) & np.isfinite(plot_df[selectivity_col])].copy()

if use_log_selectivity:
    plot_df = plot_df[plot_df[selectivity_col] > 0].copy()
    plot_df["plot_selectivity"] = np.log10(plot_df[selectivity_col])
    ylab = "Image selectivity log(FVE)"
else:
    plot_df["plot_selectivity"] = plot_df[selectivity_col]
    ylab = selectivity_col

# -----------------------
# helper: symmetric quantile binning around zero
# -----------------------
def assign_symmetric_quantile_bins(s, n_side_bins=5, include_zero_bin=False, zero_eps=1e-12):
    s = pd.Series(s).astype(float)
    out = pd.Series(np.nan, index=s.index, dtype=float)

    neg_mask = s < -zero_eps
    pos_mask = s > zero_eps
    zero_mask = (~neg_mask) & (~pos_mask)

    # negative side: bin by |slope| so -1 is closest to zero, -n is most adapting
    if neg_mask.sum() > 0:
        neg_mag = s.loc[neg_mask].abs()
        neg_bins = pd.qcut(neg_mag, q=min(n_side_bins, neg_mask.sum()), labels=False, duplicates="drop")
        out.loc[neg_mask] = -(neg_bins.astype(int) + 1)

    # positive side: +1 is closest to zero, +n is most facilitating
    if pos_mask.sum() > 0:
        pos_mag = s.loc[pos_mask].abs()
        pos_bins = pd.qcut(pos_mag, q=min(n_side_bins, pos_mask.sum()), labels=False, duplicates="drop")
        out.loc[pos_mask] = pos_bins.astype(int) + 1

    if include_zero_bin:
        out.loc[zero_mask] = 0

    return out.astype("Int64")

# -----------------------
# plot
# -----------------------
fig, ax = plt.subplots(figsize=figsize)

sns.despine()
ax.tick_params(axis="x", which="major", reset=True, top=False, labelsize=12)
ax.tick_params(axis="y", which="major", reset=True, right=False, labelsize=12)
for spine in ["left", "bottom"]:
    ax.spines[spine].set_linewidth(2)

qlo, qhi = band_quantiles

for depth, g in plot_df.groupby(depth_col):
    g = g.copy()
    g["sym_bin"] = assign_symmetric_quantile_bins(
        g[slope_col],
        n_side_bins=n_side_bins,
        include_zero_bin=include_zero_bin,
        zero_eps=zero_eps,
    )
    g = g.dropna(subset=["sym_bin"]).copy()

    summary = (
        g.groupby("sym_bin")
        .agg(
            y_med=("plot_selectivity", "median"),
            y_qlo=("plot_selectivity", lambda s: s.quantile(qlo)),
            y_qhi=("plot_selectivity", lambda s: s.quantile(qhi)),
            n=("plot_selectivity", "size"),
        )
        .reset_index()
        .sort_values("sym_bin")
    )

    summary = summary[summary["n"] >= min_bin_n].copy()
    if len(summary) == 0:
        continue

    x = summary["sym_bin"].astype(int).to_numpy()

    ax.plot(
        x,
        summary["y_med"].to_numpy(),
        marker="o",
        lw=2,
        label=f"{int(depth)} µm",
        color=DEPTH_PALETTE[depth],
    )
    ax.fill_between(
        x,
        summary["y_qlo"].to_numpy(),
        summary["y_qhi"].to_numpy(),
        alpha=0.15,
        color=DEPTH_PALETTE[depth],
    )

xticks = list(range(-n_side_bins, 0))
if include_zero_bin:
    xticks += [0]
xticks += list(range(1, n_side_bins + 1))

ax.set_xticks(xticks)
ax.set_xlabel('Binned response slope')
ax.set_ylabel('Image selectivity (FVE)')
ax.legend(fontsize=12, frameon=False, ncols=2,columnspacing=1)
plt.tight_layout()
filen = 'Binned_resp_slope'
save_figure(fig,os.path.join(savepath,filen),formats=['.pdf','.png'],dpi=300)

In [ ]:
df.keys()

In [ ]:
fig,ax=plt.subplots()

x = plot_df['log10_image_rms_dff']
y = plot_df['median_early_slope']

sns.scatterplot(data=dft,x=x,y=y,hue='depth_um',palette=DEPTH_PALETTE,edgecolor=None,alpha=0.55)

In [ ]:
for depth in dft['depth_um'].unique():
    df = dft[dft['depth_um']==depth]
    fig,ax=plt.subplots()
    ax.scatter(df['max_mag_early_slope'].clip(upper=200),df['log10_image_rms_dff'])